# Exp 31 - Cost-to-serve (development, $0, evaluator-side arithmetic on Exp 30's measured numbers)
**31A** measured system cost, split by path. **31B** human-review sensitivity (3 assumption scenarios). **31C** incorrect-decision cost, split by error type and consequence -- not one flat "wrong decision" cost, since Exp 30 has 0 false approvals and the 27 errors have very different business consequences.
All dollar-per-error and dollar-per-review figures below are **modelled assumptions, clearly labelled**, never presented as measured savings.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parents[0]; sys.path.insert(0, str(ROOT))
from src import config as C, evaluate
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 100, 'display.max_rows', 200)
gt = evaluate.load_gt()
recs = [json.loads(l) for l in (C.RESULTS/'development'/'exp30_selective_router'/'predictions.jsonl').read_text().splitlines()]
summ = json.loads((C.RESULTS/'development'/'exp30_selective_router'/'summary.json').read_text())
out = json.loads((C.RESULTS/'development'/'exp30_selective_router'/'predictions.jsonl').read_text().splitlines()[0])  # sanity peek
print(summ['correct'], '/', summ['n'], '| HRR', summ['human_review_rate'])

43 / 70 | HRR 0.1857


## 31A: measured system cost, by path

In [2]:
from src import resolver, llm_exp
cases = llm_exp.cases_for('DEVELOPMENT')
det_recs = {r['case_id']: r for r in recs}
# path is not saved in predictions.jsonl (only in resolver.resolve_batch's in-memory dict), so recompute the deterministic/residual split the same way Exp 30 did (same visible-only signal, $0, no new LLM calls)
det = {cc['case_id']: resolver.deterministic(cc) for cc in cases}
det_ids = [cid for cid, (d, conc) in det.items() if conc]; res_ids = [cid for cid, (d, conc) in det.items() if not conc]
det_cost = sum(det_recs[c]['model_cost_usd'] for c in det_ids); res_cost = sum(det_recs[c]['model_cost_usd'] for c in res_ids)
det_lat = [det_recs[c]['latency_ms'] for c in det_ids]; res_lat = [det_recs[c]['latency_ms'] for c in res_ids]
import statistics as st
a31 = pd.DataFrame([
    {'path': 'deterministic', 'n': len(det_ids), 'cost_per_claim_usd': round(det_cost/len(det_ids), 6), 'avg_latency_ms': round(st.mean(det_lat), 2) if det_lat else 0},
    {'path': 'llm_residual', 'n': len(res_ids), 'cost_per_claim_usd': round(res_cost/len(res_ids), 6), 'avg_latency_ms': round(st.mean(l for l in res_lat if l), 2) if any(res_lat) else 'not logged per-call (see Exp 20 pilot: ~1-2s/turn)'},
])
blended_cost = summ['total_cost_usd'] / summ['n']
print(a31.to_string(index=False))
print(f"\nblended cost per claim: ${blended_cost:.6f} | % claims invoking the LLM: {len(res_ids)}/{summ['n']} = {len(res_ids)/summ['n']:.0%}")

         path  n  cost_per_claim_usd  avg_latency_ms
deterministic 34            0.000000            0.00
 llm_residual 36            0.000832         2076.83

blended cost per claim: $0.000428 | % claims invoking the LLM: 36/70 = 51%


In [3]:
for n in (1000, 10000, 100000):
    print(f'{n:>7,} claims/month -> AI cost ${blended_cost*n:,.2f} (deterministic-path claims cost $0 each; all AI cost comes from the {len(res_ids)/summ["n"]:.0%} of claims routed to the LLM)')

  1,000 claims/month -> AI cost $0.43 (deterministic-path claims cost $0 each; all AI cost comes from the 51% of claims routed to the LLM)
 10,000 claims/month -> AI cost $4.28 (deterministic-path claims cost $0 each; all AI cost comes from the 51% of claims routed to the LLM)
100,000 claims/month -> AI cost $42.81 (deterministic-path claims cost $0 each; all AI cost comes from the 51% of claims routed to the LLM)


## 31B: human-review sensitivity (three labelled assumption scenarios)
Layer 1 = measured AI cost (31A). Layer 2 = Human Review Rate x an assumed manual-review cost. HRR here is Exp 30's measured 18.6%.

In [4]:
HRR = summ['human_review_rate']
SCENARIOS = {'low (a quick 5-minute check, junior finance staff)': 5, 'medium (a 15-minute review with some back-and-forth)': 15, 'high (a 30-minute review needing a manager)': 30}
rows = []
for name, dollars in SCENARIOS.items():
    for n in (1000, 10000, 100000):
        rows.append({'scenario (ASSUMPTION)': name, 'claims/month': n, 'AI cost': round(blended_cost*n, 2), 'human_review_cost (ASSUMPTION)': round(HRR*n*dollars, 2), 'total_layer1+2 (MODELLED)': round(blended_cost*n + HRR*n*dollars, 2)}
    )
b31 = pd.DataFrame(rows)
b31

,scenario (ASSUMPTION),claims/month,AI cost,human_review_cost (ASSUMPTION),total_layer1+2 (MODELLED)
0,"low (a quick 5-minute check, junior finance staff)",1000,0.43,928.5,928.93
1,"low (a quick 5-minute check, junior finance staff)",10000,4.28,9285.0,9289.28
2,"low (a quick 5-minute check, junior finance staff)",100000,42.81,92850.0,92892.81
3,medium (a 15-minute review with some back-and-forth),1000,0.43,2785.5,2785.93
4,medium (a 15-minute review with some back-and-forth),10000,4.28,27855.0,27859.28
5,medium (a 15-minute review with some back-and-forth),100000,42.81,278550.0,278592.81
6,high (a 30-minute review needing a manager),1000,0.43,5571.0,5571.43
7,high (a 30-minute review needing a manager),10000,4.28,55710.0,55714.28
8,high (a 30-minute review needing a manager),100000,42.81,557100.0,557142.81


## 31C: incorrect-decision cost, by error type and consequence -- not one flat cost
Exp 30 has **0 false approvals**; the story is not "39% error rate = 39% financial loss". Each of the 27 dev errors is categorised by its actual consequence, with a distinct, labelled assumed cost per category.

In [5]:
from collections import Counter
errs = [(gt[r['case_id']]['expected_decision'], r['predicted_decision']) for r in recs if r['predicted_decision'] != gt[r['case_id']]['expected_decision']]
CATS = {
    ('APPROVE','REJECT'): ('false_rejection', 'a valid claim wrongly rejected: employee friction/appeal'),
    ('APPROVE','REQUEST_INFORMATION'): ('unnecessary_request_info', 'a valid claim delayed by an unneeded information request'),
    ('REJECT','REQUEST_INFORMATION'): ('unnecessary_request_info', 'a non-compliant claim delayed rather than rejected outright (caught later, low risk)'),
    ('ESCALATE','REJECT'): ('missed_escalate', 'a case needing Finance judgement was auto-rejected instead: compliance/precedent risk'),
    ('REQUEST_INFORMATION','REJECT'): ('missed_request_info', 'a claim that only needed clarification was denied outright: risk of wrongly losing a legitimate claim'),
}
cat_counts = Counter(CATS[e][0] for e in errs)
ASSUMED_COST = {'false_approval': 200, 'false_rejection': 30, 'missed_escalate': 100, 'missed_request_info': 40, 'unnecessary_request_info': 8}  # USD per instance, ASSUMPTIONS
fa_count = sum(1 for r in recs if r['predicted_decision']=='APPROVE' and gt[r['case_id']]['expected_decision']!='APPROVE')
cat_counts['false_approval'] = fa_count
c31 = pd.DataFrame([{'category': k, 'count_of_27_dev_errors': v, 'assumed_cost_per_instance_USD (ASSUMPTION)': ASSUMED_COST[k], 'modelled_total_USD': v*ASSUMED_COST[k]} for k, v in cat_counts.items()])
c31 = c31.sort_values('modelled_total_USD', ascending=False)
print(c31.to_string(index=False))
print(f"\ntotal modelled error cost across the 27 dev errors: ${c31.modelled_total_USD.sum()} (0 false approvals contribute $0 -- the largest per-instance risk category never occurred on dev)")

                category  count_of_27_dev_errors  assumed_cost_per_instance_USD (ASSUMPTION)  modelled_total_USD
         missed_escalate                       4                                         100                 400
         false_rejection                       8                                          30                 240
unnecessary_request_info                      13                                           8                 104
     missed_request_info                       2                                          40                  80
          false_approval                       0                                         200                   0

total modelled error cost across the 27 dev errors: $824 (0 false approvals contribute $0 -- the largest per-instance risk category never occurred on dev)


In [6]:
err_rate_by_cat = {k: v/summ['n'] for k, v in cat_counts.items()}
rows = []
for n in (1000, 10000, 100000):
    row = {'claims/month': n}
    for k, rate in err_rate_by_cat.items():
        row[k+'_modelled_cost'] = round(rate*n*ASSUMED_COST[k], 2)
    row['total_modelled_error_cost'] = round(sum(rate*n*ASSUMED_COST[k] for k, rate in err_rate_by_cat.items()), 2)
    rows.append(row)
pd.DataFrame(rows)

,claims/month,unnecessary_request_info_modelled_cost,missed_escalate_modelled_cost,missed_request_info_modelled_cost,false_rejection_modelled_cost,false_approval_modelled_cost,total_modelled_error_cost
0,1000,1485.71,5714.29,1142.86,3428.57,0.0,11771.43
1,10000,14857.14,57142.86,11428.57,34285.71,0.0,117714.29
2,100000,148571.43,571428.57,114285.71,342857.14,0.0,1177142.86


## What was done and what it means
- **31A, measured (not modelled):** the deterministic path costs $0/claim with effectively instant latency (34/70 claims). The LLM-residual path costs **$0.000832/claim**, averaging **2,077 ms** per call. Blended across all 70 claims: **$0.000428/claim**, with **51% of claims routed to the LLM.** Scaled: $0.43 at 1,000 claims/month, $4.28 at 10,000, $42.81 at 100,000. **The AI cost is trivial at every scale tested** -- it is not the cost driver of this system.
- **31B, human review (three labelled assumptions):** even the "low" scenario (a 5-minute check) puts human-review cost at **$928.50/month at 1,000 claims**, rising to **$92,850/month at 100,000** -- roughly **2,100x the AI cost at every scale**, because HRR (18.6%) multiplies a per-instance human cost that is orders of magnitude larger than a fraction of a cent. The "high" scenario reaches **$557,100/month at 100,000 claims/month**. **Human review, not the model, dominates the cost-to-serve story at every scale and every assumption tested.** This is the single most important economic finding of Exp 31: reducing Human Review Rate (which is exactly what Exp 29's abstention analysis was about) matters far more to the bottom line than optimising AI cost further.
- **31C, incorrect-decision cost, split by consequence, not one flat rate:** of the 27 dev errors, **missed_escalate is the single largest category by modelled cost ($400 total, 4 instances at an assumed $100 each)** -- these are cases needing Finance judgement that were auto-rejected instead, the riskiest category since it bypasses a required review rather than just causing friction. **false_rejection is second ($240, 8 instances)** -- valid claims wrongly denied, a real employee-relations cost. **unnecessary_request_info is the most frequent (13 instances) but individually cheapest ($8 each, $104 total)** -- pure delay, no wrong final outcome assumed. **false_approval contributes $0, because Exp 30 had zero of them** -- the highest per-instance assumed cost ($200) never materialised on this dev set, which is exactly the point of reporting by category rather than a flat "39% error rate = 39% loss" claim, which would have been misleading. Scaled to 100,000 claims/month, total modelled error cost is about **$1.18M/month** under these assumptions, still smaller than the "high" human-review scenario alone ($557k) is comparable in order of magnitude but assumption-driven in both directions -- neither should be read as a precise forecast.
- **The headline, honestly stated:** at every scale and under every assumption tested, **AI cost is negligible, human review cost is the dominant and controllable lever, and error cost is real but concentrated in a small number of categories (missed escalation, false rejection) rather than spread evenly.** None of the dollar figures here are measured; they are modelled from the plan's own labelled-assumption discipline, and should be presented as such, never as "measured savings."
- **Cost of running this experiment:** $0 (arithmetic over Exp 30's already-measured numbers). Total spend unchanged at $2.139 of $3.50.
- **Decision:** the economic case for the selective architecture rests on keeping Human Review Rate low without letting error cost concentrate in the riskiest categories (missed_escalate, false_rejection) -- both point back to Exp 30's own finding that the LLM-residual path (36.1% accuracy) is the weak link worth improving before anything else in this system. Next: Exp 32, the frozen final test, run once.